# CAPO Basics: Understanding Covariance-Aware Weighting

This notebook introduces the core concepts of CAPO (Covariance-Aware Policy Optimization).

## Key Idea

Standard policy gradient methods treat all trajectories equally. But when variance depends on trajectory length, this is suboptimal. CAPO models:

$$v_i = \sigma^2 L_i^\beta s(L_i; \xi)$$

where:
- $\beta$ is the length exponent
- $s(L; \xi)$ captures within-trajectory dependence

And uses precision-optimal weights: $w_i \propto L_i^{-\beta} / s(L_i; \xi)$

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from capo.eb_core import kband_weights, s_kband

## 1. Length-Based Weighting

Let's see how different $\beta$ values affect trajectory weights.

In [ ]:
# Trajectories of different lengths
L = torch.tensor([16.0, 32.0, 64.0, 128.0, 256.0])

# Compare different beta values
betas = [0.0, 0.5, 1.0, 1.5]

fig, ax = plt.subplots(figsize=(10, 5))

for beta in betas:
    s, w = kband_weights(L=L, beta=beta, rho=0.0, eta=1.0, k=0)
    ax.plot(L.numpy(), w.numpy(), 'o-', label=f'β={beta}')

ax.set_xlabel('Trajectory Length')
ax.set_ylabel('Weight')
ax.set_title('CAPO Weights vs Length for Different β')
ax.legend()
ax.set_xscale('log', base=2)
plt.tight_layout()
plt.show()

print("Key insight: Higher β → shorter trajectories get more weight")

## 2. The Dependence Factor $s(L; \xi)$

When tokens within a trajectory are correlated, we need the k-banded dependence factor.

In [ ]:
# Compare s(L) for different correlation strengths
L_range = torch.linspace(10, 500, 50)
rhos = [0.0, 0.3, 0.5, 0.7]

fig, ax = plt.subplots(figsize=(10, 5))

for rho in rhos:
    s = s_kband(L_range, rho=rho, k=20, eta=1.0)
    ax.plot(L_range.numpy(), s.numpy(), label=f'ρ={rho}')

ax.set_xlabel('Trajectory Length')
ax.set_ylabel('s(L; ρ, k=20)')
ax.set_title('Dependence Factor for Different Correlation Strengths')
ax.legend()
plt.tight_layout()
plt.show()

print("Key insight: Higher ρ → higher variance inflation from dependence")

## 3. Toy Example from the Paper

Three trajectories: L = [16, 64, 256], gradients g = [0.8, 1.0, 1.1]

In [ ]:
L = torch.tensor([16.0, 64.0, 256.0])
g = torch.tensor([0.8, 1.0, 1.1])

print("Trajectory data:")
for i in range(3):
    print(f"  Trajectory {i+1}: L={int(L[i])}, g={g[i]:.1f}")

print("\nWeighted means for different β:")
for beta in [0.0, 0.5, 1.0]:
    s, w = kband_weights(L=L, beta=beta, rho=0.0, eta=1.0, k=0)
    m = (w * g).sum().item()
    print(f"  β={beta}: w={[f'{x:.3f}' for x in w.tolist()]}, m={m:.3f}")

print("\nNote: β=0 gives equal weights (standard GRPO)")
print("      β=1 gives ΔL normalization (weights ∝ 1/L)")

## Summary

- **β = 0**: Equal weighting (standard GRPO)
- **β = 1**: ΔL normalization (inverse length weighting)
- **ρ > 0**: Accounts for within-trajectory token dependence

CAPO's Empirical Bayes procedure estimates these parameters from data!